# Merge and Prepare the Dudley Datasets

## Purpose
Combine the five cleaned datasets to explore local factors associated
with economic activity and possible employment support needs in Dudley.

## Objectives
- Verify geographic compatibility between the datasets.
- Select relevant columns and distinguish their source years.
- Merge compatible areas using LSOA codes.
- Check merge coverage, missing values and duplicate records.
- Calculate percentages using the appropriate population or
  household denominator from each source.
- Save the combined dataset for later analysis.

## Inputs
Five cleaned CSV files from `cleaned_datasets`:
- `dudley_economic_activity_cleaned.csv`
- `dudley_qualifications_cleaned.csv`
- `dudley_disability_16_cleaned.csv`
- `dudley_car_availability_cleaned.csv`
- `dudley_iod_cleaned.csv`

## Planned Output
`cleaned_datasets/dudley_combined_prepared.csv`

## Geographic and Time Coverage
The Census datasets contain 203 Dudley areas. The deprivation
dataset contains 201 areas using 2011 geography.

The datasets share 199 LSOA codes. The planned merge will use
matching areas after boundary compatibility has been checked.
Any exclusions will be documented.

Deprivation data comes from the 2019 release and provides historical
context alongside Census 2021 data.

## Ethical Considerations
Area-level patterns do not establish individual circumstances or
prove causes of unemployment. Disability, qualifications and car
availability must be interpreted respectfully and in context.

Different source dates, population definitions and excluded areas
will be considered when interpreting results. The original cleaned
datasets will remain unchanged.

## Additional Comments
ChatGPT assisted with planning, code development and documentation.
This notebook covers merging and preparation; EDA, hypothesis
testing and modelling will follow separately.

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [2]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [3]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [4]:
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers'

## 1. Load the Cleaned Datasets

Load each cleaned CSV into a separate dataframe.
Display their dimensions and column names to confirm the available
fields before selecting variables or calculating percentages.

No source files will be changed.

In [5]:
# Developed with ChatGPT assistance to load the cleaned datasets.
import pandas as pd

# Map each dataset to its saved CSV filename.
dataset_files = {
    "economic": "dudley_economic_activity_cleaned.csv",
    "qualifications": "dudley_qualifications_cleaned.csv",
    "disability": "dudley_disability_16_cleaned.csv",
    "car": "dudley_car_availability_cleaned.csv",
    "iod": "dudley_iod_cleaned.csv"
}

# Load each file separately, preserving the saved source files.
datasets = {
    name: pd.read_csv(f"cleaned_datasets/{filename}")
    for name, filename in dataset_files.items()
}

# Inspect dimensions and exact column names before merging.
for name, dataframe in datasets.items():
    print(f"\n{name.upper()}")
    print(f"Rows: {dataframe.shape[0]}, Columns: {dataframe.shape[1]}")
    print(dataframe.columns.tolist())


ECONOMIC
Rows: 203, Columns: 11
['Original_Area', 'Total_Aged_16_Plus', 'Economically_Active_Excl_FT_Students', 'Employed_Excl_FT_Students', 'Unemployed_Excl_FT_Students', 'Inactive_Student', 'Inactive_Home_Or_Family', 'Inactive_Long_Term_Sick_Or_Disabled', 'Inactive_Other', 'LSOA_Code', 'LSOA_Name']

QUALIFICATIONS
Rows: 203, Columns: 10
['LSOA_Name', 'LSOA_Code', 'Total_Aged_16_Plus', 'No_Qualifications', 'Level_1_And_Entry', 'Level_2', 'Apprenticeship', 'Level_3', 'Level_4_And_Above', 'Other_Qualifications']

DISABILITY
Rows: 203, Columns: 7
['LSOA_Name', 'LSOA_Code', 'Total_16_Plus', 'Disabled_16_Plus', 'Limited_A_Lot_16_Plus', 'Limited_A_Little_16_Plus', 'Not_Disabled_16_Plus']

CAR
Rows: 203, Columns: 7
['LSOA_Name', 'LSOA_Code', 'Total_Households', 'No_Car_Van_Households', 'One_Car_Van_Households', 'Two_Cars_Vans_Households', 'Three_Plus_Cars_Vans_Households']

IOD
Rows: 201, Columns: 57
['LSOA_code_2011', 'LSOA_name_2011', 'Local_Authority_District_code_2019', 'Local_Authority

### Findings
- All five cleaned datasets loaded successfully.
- Economic activity contains 203 rows and 11 columns.
- Qualifications contains 203 rows and 10 columns.
- Disability and car availability each contain 203 rows and 7 columns.
- Deprivation contains 201 rows and 57 columns.
- The four Census datasets use `LSOA_Code`; deprivation uses
  `LSOA_code_2011`.
- Population totals have overlapping names across datasets.
  They must remain distinguishable so percentages use the
  correct source-specific denominator.
- The economic activity file contains selected categories,
  including employment measures excluding full-time students.
  These must not be labelled as covering all employment.
- No datasets have been merged or modified.

---

## 2. Check Merge Keys and Shared Coverage

Check that each dataset has complete, unique LSOA codes so merging
will not create duplicate area records.

Confirm whether the four Census datasets contain the same codes
and count the codes shared by all five datasets.

Matching codes establish join coverage. Compatibility between
2011 and 2021 boundaries still requires a separate check.

In [8]:
# Developed with ChatGPT assistance to check merge keys and coverage.

# Identify the geographic key used by each dataset.
key_columns = {
    name: "LSOA_code_2011" if name == "iod" else "LSOA_Code"
    for name in datasets
}

# Check missing and duplicate keys before merging.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html
# https://pandas.pydata.org/docs/reference/api/pandas.Series.duplicated.html
check_results = []

for name, dataframe in datasets.items():
    codes = dataframe[key_columns[name]]

    check_results.append({
        "Dataset": name,
        "Rows": len(dataframe),
        "Missing_codes": codes.isna().sum(),
        "Duplicate_codes": codes.duplicated().sum()
    })

key_checks = pd.DataFrame(check_results)
display(key_checks)

# Remove missing codes and create sets of unique codes.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.dropna.html
# https://docs.python.org/3/library/stdtypes.html#set-types-set-frozenset
code_sets = {}

for name, dataframe in datasets.items():
    codes = dataframe[key_columns[name]].dropna()
    code_sets[name] = set(codes)

# Check whether the Census datasets cover the same areas.
# Reference:
# https://docs.python.org/3/library/functions.html#all
census_names = [
    "economic",
    "qualifications",
    "disability",
    "car"
]

same_census_codes = all(
    code_sets[name] == code_sets["economic"]
    for name in census_names
)

# Find codes present in every dataset.
# Reference:
# https://docs.python.org/3/library/stdtypes.html#frozenset.intersection
shared_codes = set.intersection(*code_sets.values())

print("All four Census code sets match:", same_census_codes)
print("Codes shared by all five datasets:", len(shared_codes))

,Dataset,Rows,Missing_codes,Duplicate_codes
0,economic,203,0,0
1,qualifications,203,0,0
2,disability,203,0,0
3,car,203,0,0
4,iod,201,0,0


All four Census code sets match: True
Codes shared by all five datasets: 199


### Findings
- All five datasets have complete, unique LSOA codes.
- No missing or duplicate codes were found.
- The four Census datasets contain the same 203 area codes.
- A total of 199 codes are shared by all five datasets.
- The keys support one-to-one joins without duplicate matches.
- Compatibility between the 2011 and 2021 boundaries still
  needs verification before merging.
- No records have been removed or merged.

---

## 3. Load the Geographic Lookup

Load the official ONS lookup linking 2011 and 2021 LSOA areas.
Inspect its columns and preview five records before using its
change indicator to check boundary compatibility.

This lookup supports geographic validation; it will not add
analytical features to the combined dataset.

In [9]:
# Developed with ChatGPT assistance to inspect the ONS lookup.

# Load the downloaded lookup without changing its contents.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
lookup_path = "raw_datasets/lsoa_2011_2021_lookup.csv"
lookup_df = pd.read_csv(lookup_path)

print(f"Rows: {lookup_df.shape[0]}")
print(f"Columns: {lookup_df.shape[1]}")

# Display exact column names to guide the next check.
for column in lookup_df.columns:
    print(column)

# Preview five records.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html
display(lookup_df.head())

Rows: 35796
Columns: 9
LSOA11CD
LSOA11NM
LSOA21CD
LSOA21NM
CHGIND
LAD22CD
LAD22NM
LAD22NMW
ObjectId


,LSOA11CD,LSOA11NM,LSOA21CD,LSOA21NM,CHGIND,LAD22CD,LAD22NM,LAD22NMW,ObjectId
0,E01031349,Adur 001A,E01031349,Adur 001A,U,E07000223,Adur,NaN,1
1,E01031350,Adur 001B,E01031350,Adur 001B,U,E07000223,Adur,NaN,2
2,E01031351,Adur 001C,E01031351,Adur 001C,U,E07000223,Adur,NaN,3
3,E01031352,Adur 001D,E01031352,Adur 001D,U,E07000223,Adur,NaN,4
4,E01031370,Adur 001E,E01031370,Adur 001E,U,E07000223,Adur,NaN,5


### Findings
- The lookup loaded successfully with 35,796 rows and 9 columns.
- `LSOA11CD` and `LSOA21CD` contain the geographic codes
  for 2011 and 2021.
- `CHGIND` records the type of geographic change.
- The five previewed records have indicator `U` (unchanged)
  and identical codes in both years.
- These preview records are for Adur. Dudley's matching areas
  still need to be checked.

---

## 4. Verify Boundary Compatibility for Shared Areas

Check the 199 shared codes against the ONS lookup.

Each shared code should have exactly one lookup record, the same
2011 and 2021 code, and change indicator `U` (unchanged).
Report any exceptions before merging.

In [10]:
# Developed with ChatGPT assistance to verify area boundaries.

# Select lookup records for the shared 2011 codes.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html
shared_lookup = lookup_df.loc[
    lookup_df["LSOA11CD"].isin(shared_codes)
].copy()

# Check that every shared code has exactly one lookup record.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html
# https://pandas.pydata.org/docs/reference/api/pandas.Series.reindex.html
lookup_counts = shared_lookup["LSOA11CD"].value_counts()
lookup_counts = lookup_counts.reindex(
    sorted(shared_codes),
    fill_value=0
)

print("Shared codes checked:", len(shared_codes))
print("Codes missing from lookup:", (lookup_counts == 0).sum())
print("Codes with multiple records:", (lookup_counts > 1).sum())

# Flag changed areas or different codes between the two years.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.eq.html
same_code = shared_lookup["LSOA11CD"].eq(
    shared_lookup["LSOA21CD"]
)
unchanged = shared_lookup["CHGIND"].eq("U")
exceptions = shared_lookup.loc[~(same_code & unchanged)]

print("Records failing boundary checks:", len(exceptions))
print("\nChange indicators:")
print(shared_lookup["CHGIND"].value_counts(dropna=False))

# Preview any exceptions, showing only the relevant fields.
display(
    exceptions[
        ["LSOA11CD", "LSOA21CD", "CHGIND", "LAD22NM"]
    ].head()
)

Shared codes checked: 199
Codes missing from lookup: 0
Codes with multiple records: 0
Records failing boundary checks: 0

Change indicators:
CHGIND
U    199
Name: count, dtype: int64


,LSOA11CD,LSOA21CD,CHGIND,LAD22NM


### Findings
- All 199 shared codes were found in the official ONS lookup.
- Each code has exactly one lookup record.
- All records have identical 2011 and 2021 LSOA codes.
- All 199 records have change indicator `U` (unchanged).
- No boundary-check exceptions were found.
- The shared areas are geographically compatible for merging.
  Differences in dataset dates still apply.

---

## 5. Check Percentage Denominators

Check that each Census dataset's population or household total
is present and greater than zero before calculating percentages.

Use the total from the same dataset as the category being measured.
Do not substitute totals from another source, even when the
population labels are similar.

The deprivation rates are already supplied as proportions and
will be handled separately.

In [11]:
# Developed with ChatGPT assistance to check denominators.

# Use each dataset's own population or household total.
denominator_columns = {
    "economic": "Total_Aged_16_Plus",
    "qualifications": "Total_Aged_16_Plus",
    "disability": "Total_16_Plus",
    "car": "Total_Households"
}

# Check missing, zero and negative denominator values.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html
# https://pandas.pydata.org/docs/reference/api/pandas.Series.le.html
denominator_results = []

for name, column in denominator_columns.items():
    totals = datasets[name][column]

    denominator_results.append({
        "Dataset": name,
        "Denominator": column,
        "Minimum": totals.min(),
        "Maximum": totals.max(),
        "Missing": totals.isna().sum(),
        "Zero_or_negative": totals.le(0).sum()
    })

denominator_checks = pd.DataFrame(denominator_results)
display(denominator_checks)

,Dataset,Denominator,Minimum,Maximum,Missing,Zero_or_negative
0,economic,Total_Aged_16_Plus,986,2166,0,0
1,qualifications,Total_Aged_16_Plus,985,2169,0,0
2,disability,Total_16_Plus,985,2168,0,0
3,car,Total_Households,460,1183,0,0


### Findings
- All four denominator columns contain positive values.
- No missing, zero or negative denominators were found.
- Population totals vary slightly between the Census datasets.
  Each percentage will therefore use its own source's total.
- Car availability percentages will use household totals,
  rather than population totals.
- These totals support calculating shares of residents aged
  16 and over, or households, as appropriate.
- An unemployment rate would require a separate economically
  active denominator; it is not the same as a population share.

---

# Section 2

Section 2 content

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
